# PrioritAI — Conformal Calibration Notebook
## Generating `q_hat` for the ResNet50 Damage Classifier

---

### What this notebook does

This is an **offline calibration step**. It does not retrain or modify the model.

Using a labeled calibration dataset, it computes `q_hat` — the conformal threshold that gives a **coverage guarantee**: with `alpha = 0.05`, the prediction set will contain the true damage class at least **95% of the time** on data drawn from the same distribution.

### Architecture context

```
Frontend → Backend (FastAPI) → ML Service (FastAPI + Keras)
                                         ↑
                               [Conformal wrapper — future]
                               reads q_hat.json at startup
```

### Class ordering

The model was trained with folders sorted **alphabetically**:

| Index | Class  | damage_score |
|-------|--------|--------------|
| 0     | heavy  | 7            |
| 1     | light  | 3            |
| 2     | medium | 5            |

### Output

`q_hat.json` saved to Google Drive — loaded by the runtime service in a future step.

---
## Step 1 — Mount Google Drive

All paths below are relative to your Drive root. Edit the `DRIVE_ROOT` variable in Step 2 if your files are in a subfolder.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

---
## Step 2 — Configuration

Edit these paths to match your Google Drive layout before running the rest of the notebook.

```
MyDrive/
├── rocket_damage_resnet50_v2.keras   ← MODEL_PATH
├── labeled_data/                      ← DATA_ROOT
│   ├── heavy/
│   ├── light/
│   └── medium/
└── conformal_output/
    └── q_hat.json                     ← OUTPUT_PATH (created automatically)
```

In [ ]:
# ── Google Drive paths ────────────────────────────────────────────────────────
DRIVE_ROOT  = "/content/drive/MyDrive"
MODEL_PATH  = f"{DRIVE_ROOT}/rocket_damage_resnet50_v2.keras"
DATA_ROOT   = f"{DRIVE_ROOT}/labeled_data"
OUTPUT_PATH = f"{DRIVE_ROOT}/conformal_output/q_hat.json"

# ── Must match production ml-service/app/inference.py ────────────────────────
IMG_SIZE    = (224, 224)      # resize target
CLASS_NAMES = ["heavy", "light", "medium"]  # alphabetical = model index order

# ── Conformal calibration ─────────────────────────────────────────────────────
ALPHA       = 0.05            # target miscoverage rate → 95 % coverage guarantee
BATCH_SIZE  = 32              # images per model.predict() call

print("Configuration:")
print(f"  MODEL_PATH  : {MODEL_PATH}")
print(f"  DATA_ROOT   : {DATA_ROOT}")
print(f"  OUTPUT_PATH : {OUTPUT_PATH}")
print(f"  IMG_SIZE    : {IMG_SIZE}")
print(f"  CLASS_NAMES : {CLASS_NAMES}")
print(f"  ALPHA       : {ALPHA}")

---
## Step 3 — Imports

All libraries are pre-installed in Google Colab. No `pip install` needed.

In [ ]:
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import tensorflow as tf
from PIL import Image

print(f"TensorFlow : {tf.__version__}")
print(f"NumPy      : {np.__version__}")
print(f"Pillow     : {Image.__version__}")

---
## Step 4 — Load the Keras Model

The model is loaded identically to production `preload_model()` in `ml-service/app/inference.py`:

```python
model = tf.keras.models.load_model(MODEL_PATH)
```

A warmup predict runs immediately after loading to force graph compilation, matching runtime behaviour.

In [ ]:
print(f"Loading model from:\n  {MODEL_PATH}\n")

if not Path(MODEL_PATH).exists():
    raise FileNotFoundError(
        f"Model not found at {MODEL_PATH}\n"
        "Check that MODEL_PATH in Step 2 points to the correct Drive location."
    )

model = tf.keras.models.load_model(MODEL_PATH)

# Warmup pass — identical to production preload_model()
dummy = np.zeros((1, *IMG_SIZE, 3), dtype=np.float32)
_ = model.predict(dummy, verbose=0)
print("Warmup complete.\n")

model.summary()

---
## Step 5 — Preprocessing Pipeline

This function replicates the **exact** preprocessing from `ml-service/app/inference.py`:

```python
img = Image.open(io.BytesIO(image_bytes)).convert("RGB")
img = img.resize(IMG_SIZE)                          # 224 × 224
arr = np.array(img, dtype=np.float32) / 255.0      # normalise [0, 1]
arr = np.expand_dims(arr, axis=0)                   # add batch dim
```

The only difference here is that we read from a file path instead of bytes, and we do not add the batch dimension yet (that happens during batched inference in Step 6).

In [ ]:
def preprocess_image(image_path: str) -> np.ndarray:
    """
    Identical preprocessing to production inference.py.

    Steps:
      1. Open image file and convert to RGB (drops alpha, handles grayscale)
      2. Resize to IMG_SIZE = (224, 224)
      3. Normalise pixel values from [0, 255] to [0.0, 1.0]

    Returns:
        np.ndarray of shape (224, 224, 3), dtype float32.
        No batch dimension — caller adds it during batching.
    """
    img = Image.open(image_path).convert("RGB")
    img = img.resize(IMG_SIZE)
    arr = np.array(img, dtype=np.float32) / 255.0
    return arr

print("preprocess_image() defined.")
print(f"Output shape per image: {IMG_SIZE + (3,)}")

---
## Step 6 — Load Calibration Dataset

Images are loaded from the three class folders. Folder names are sorted **alphabetically** to assign class indices — this must match how the model was trained:

| Folder | Index |
|--------|-------|
| heavy  | 0     |
| light  | 1     |
| medium | 2     |

Supported image extensions: `.jpg`, `.jpeg`, `.png`, `.bmp`, `.webp`

In [ ]:
CLASS_TO_IDX     = {name: idx for idx, name in enumerate(CLASS_NAMES)}
VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

image_paths  = []   # List[Path]
image_labels = []   # List[int]

print("Scanning calibration dataset...")
print(f"  Root    : {DATA_ROOT}")
print(f"  Classes : {CLASS_NAMES}  (indices 0-{len(CLASS_NAMES)-1})\n")

for class_name in CLASS_NAMES:
    class_dir = Path(DATA_ROOT) / class_name

    if not class_dir.is_dir():
        print(f"  [WARN] Folder not found, skipping: {class_dir}")
        continue

    files = sorted(
        f for f in class_dir.iterdir()
        if f.suffix.lower() in VALID_EXTENSIONS
    )

    image_paths.extend(files)
    image_labels.extend([CLASS_TO_IDX[class_name]] * len(files))

    print(f"  [{CLASS_TO_IDX[class_name]}] {class_name:8s}  →  {len(files):4d} images")

print(f"\nTotal: {len(image_paths)} images")

if len(image_paths) == 0:
    raise RuntimeError(
        "No images found. Check that DATA_ROOT in Step 2 points to the "
        "correct Drive folder and that image files exist inside the class subfolders."
    )

---
## Step 7 — Run Model Inference on All Calibration Images

Images are processed in batches of `BATCH_SIZE = 32` to avoid running out of memory.

The output is two arrays:
- `calibration_probs` — shape `(n, 3)`, raw softmax probabilities for each image
- `calibration_labels` — shape `(n,)`, integer ground-truth class index for each image

These are the inputs to the conformal calibration algorithm in Step 8.

In [ ]:
n           = len(image_paths)
all_probs   = []
load_errors = 0

print(f"Running inference on {n} images (batch_size={BATCH_SIZE})...\n")
t0 = time.time()

for batch_start in range(0, n, BATCH_SIZE):
    batch_paths  = image_paths[batch_start : batch_start + BATCH_SIZE]
    batch_arrays = []

    for path in batch_paths:
        try:
            arr = preprocess_image(str(path))
            batch_arrays.append(arr)
        except Exception as exc:
            print(f"  [ERROR] {path.name}: {exc}")
            load_errors += 1
            # Zero-fill so array indices stay aligned with image_labels
            batch_arrays.append(np.zeros((*IMG_SIZE, 3), dtype=np.float32))

    batch_tensor = np.stack(batch_arrays, axis=0)          # (B, 224, 224, 3)
    batch_probs  = model.predict(batch_tensor, verbose=0)  # (B, 3)
    all_probs.append(batch_probs)

    done    = min(batch_start + BATCH_SIZE, n)
    elapsed = time.time() - t0
    pct     = done / n * 100
    print(f"  [{done:4d}/{n}]  {pct:5.1f}%   {elapsed:5.1f}s elapsed")

calibration_probs  = np.concatenate(all_probs, axis=0)        # (n, 3)
calibration_labels = np.array(image_labels, dtype=np.int32)   # (n,)

print(f"\nInference complete.")
print(f"  calibration_probs  shape : {calibration_probs.shape}")
print(f"  calibration_labels shape : {calibration_labels.shape}")

if load_errors > 0:
    print(f"\n[WARN] {load_errors} image(s) failed to load and were replaced with zeros.")
    print("       Review the errors above — they may affect calibration quality.")

# Quick sanity: probabilities should sum to ~1 per row
row_sums = calibration_probs.sum(axis=1)
print(f"\nSoftmax row-sum check:  min={row_sums.min():.4f}  max={row_sums.max():.4f}  (expect ~1.0)")

---
## Step 8 — Conformal Calibration Algorithm

### Theory

Conformal Prediction constructs a **prediction set** instead of a single label. The set is guaranteed to contain the true class with at least `1 - alpha` probability, without any distributional assumptions beyond exchangeability.

**Nonconformity score** for image `i`:
$$s_i = 1 - \hat{p}_{y_i}$$
where $\hat{p}_{y_i}$ is the model's softmax probability for the **true** class $y_i$.

A high score means the model was uncertain or wrong about the true class.

**Calibration quantile:**
$$\hat{q} = \text{Quantile}\left(s_1, \ldots, s_n,\ \frac{\lceil (n+1)(1-\alpha) \rceil}{n}\right)$$

The `+1` finite-sample correction ensures exact marginal coverage.

**At runtime**, the prediction set for a new image is:
$$\mathcal{C}(x) = \{ k : \hat{p}_k \geq 1 - \hat{q} \}$$

### Implementation

The cell below contains the **exact algorithm** that will also be used in the runtime ML service. Do not modify it.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Exact conformal implementation.
# This code is shared between calibration (notebook) and runtime inference.
# DO NOT modify the mathematical logic.
# ─────────────────────────────────────────────────────────────────────────────

import numpy as np

def compute_conformal_threshold(
    calibration_probs, calibration_labels, alpha=0.05
):
    n = len(calibration_probs)

    if n == 0:
        return 0.75

    correct_class_probs = calibration_probs[
        np.arange(n), calibration_labels.astype(int)
    ]

    scores = 1.0 - correct_class_probs

    q_level = np.ceil((n + 1) * (1 - alpha)) / n
    q_level = min(q_level, 1.0)

    q_hat = np.quantile(scores, q_level, method="higher")

    return q_hat


def predict_conformal(model_prediction, q_hat, class_names):

    threshold = 1.0 - q_hat

    prediction_set = [
        class_names[i]
        for i, prob in enumerate(model_prediction)
        if prob >= threshold
    ]

    if not prediction_set:
        prediction_set = [class_names[np.argmax(model_prediction)]]

    return prediction_set

print("Conformal functions defined:")
print("  compute_conformal_threshold(calibration_probs, calibration_labels, alpha)")
print("  predict_conformal(model_prediction, q_hat, class_names)")

---
## Step 9 — Compute `q_hat`

Runs `compute_conformal_threshold` over the full calibration set.

In [ ]:
q_hat = compute_conformal_threshold(
    calibration_probs,
    calibration_labels,
    alpha=ALPHA,
)

threshold = 1.0 - q_hat

print("Conformal calibration complete")
print(f"  n (calibration images) : {len(calibration_probs)}")
print(f"  alpha                  : {ALPHA}  (target miscoverage rate)")
print(f"  coverage guarantee     : {(1 - ALPHA) * 100:.0f}%")
print(f"  q_hat                  : {q_hat:.6f}")
print(f"  runtime threshold      : {threshold:.6f}")
print()
print(f"Interpretation: a class is included in the prediction set")
print(f"when its softmax probability >= {threshold:.4f}")

---
## Step 10 — Save `q_hat.json`

The output file is self-documenting: it records the threshold value alongside the calibration metadata so the runtime service can validate it on load.

In [ ]:
output_dir = Path(OUTPUT_PATH).parent
output_dir.mkdir(parents=True, exist_ok=True)

payload = {
    "q_hat":          float(q_hat),
    "alpha":          ALPHA,
    "n_calibration":  int(len(calibration_probs)),
    "class_names":    CLASS_NAMES,
    "model_path":     MODEL_PATH,
    "date_saved":     datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
}

with open(OUTPUT_PATH, "w") as f:
    json.dump(payload, f, indent=2)

print(f"Saved → {OUTPUT_PATH}\n")
print(json.dumps(payload, indent=2))

---
## Step 11 — Sanity Checks

Three checks to confirm the calibration is working correctly:

1. **Sample predictions** — show prediction sets for 10 random calibration images and whether the true class is covered.
2. **Full calibration set coverage** — must be ≥ `1 - alpha` (95 %).
3. **Mean prediction set size** — indicates model certainty. Values close to 1 mean the model is confident; values close to 3 mean it is uncertain on most images.

In [ ]:
print("=" * 65)
print("SANITY CHECK 1 — prediction sets on 10 random calibration images")
print("=" * 65)

rng     = np.random.default_rng(42)
indices = rng.choice(len(calibration_probs), size=min(10, len(calibration_probs)), replace=False)

covered_sample = 0
for i in indices:
    probs      = calibration_probs[i]
    true_label = CLASS_NAMES[calibration_labels[i]]
    pred_set   = predict_conformal(probs, q_hat, CLASS_NAMES)
    is_covered = true_label in pred_set
    covered_sample += int(is_covered)
    marker = "OK" if is_covered else "MISS"
    print(
        f"  [{marker:4s}]  true={true_label:8s}  "
        f"probs={[round(p, 3) for p in probs]}  "
        f"set={pred_set}"
    )

print(f"\nSample coverage: {covered_sample}/{len(indices)} = {covered_sample / len(indices):.0%}")

# ── Full calibration set coverage ────────────────────────────────────────────
print()
print("=" * 65)
print("SANITY CHECK 2 — coverage over full calibration set")
print("=" * 65)

covered_full = sum(
    CLASS_NAMES[calibration_labels[i]] in predict_conformal(
        calibration_probs[i], q_hat, CLASS_NAMES
    )
    for i in range(len(calibration_probs))
)
full_coverage = covered_full / len(calibration_probs)
status = "PASS" if full_coverage >= (1 - ALPHA) else "FAIL"
print(f"  Coverage : {full_coverage:.4f}  (target >= {1 - ALPHA:.2f})  [{status}]")

# ── Mean prediction set size ──────────────────────────────────────────────────
print()
print("=" * 65)
print("SANITY CHECK 3 — prediction set size distribution")
print("=" * 65)

set_sizes = [
    len(predict_conformal(calibration_probs[i], q_hat, CLASS_NAMES))
    for i in range(len(calibration_probs))
]
size_counts = {s: set_sizes.count(s) for s in sorted(set(set_sizes))}

for size, count in size_counts.items():
    bar = "#" * int(count / len(set_sizes) * 40)
    print(f"  size {size}: {count:5d} images ({count / len(set_sizes):.1%})  {bar}")

print(f"\n  Mean set size : {np.mean(set_sizes):.3f}")
print(f"  (1.0 = model always certain, {len(CLASS_NAMES)}.0 = model always uncertain)")

# ── Summary ───────────────────────────────────────────────────────────────────
print()
print("=" * 65)
print("SUMMARY")
print("=" * 65)
print(f"  q_hat          : {q_hat:.6f}")
print(f"  alpha          : {ALPHA}")
print(f"  n_calibration  : {len(calibration_probs)}")
print(f"  coverage       : {full_coverage:.4f}  [{status}]")
print(f"  mean set size  : {np.mean(set_sizes):.3f}")
print(f"  output file    : {OUTPUT_PATH}")

---
## Next Steps

### What to do with `q_hat.json`

1. **Download** the file from Google Drive to your local machine.
2. **Place it** inside the ML service model directory:
   ```
   ml-service/model/q_hat.json
   ```
3. **In a future step**, the runtime `inference.py` will load `q_hat.json` at startup and use `predict_conformal()` to return prediction sets alongside the existing classification output.

### Expected `q_hat.json` structure

```json
{
  "q_hat": 0.412300,
  "alpha": 0.05,
  "n_calibration": 450,
  "class_names": ["heavy", "light", "medium"],
  "model_path": "/content/drive/MyDrive/rocket_damage_resnet50_v2.keras",
  "date_saved": "2026-05-25T10:00:00Z"
}
```

### Important notes

- The `class_names` list order in `q_hat.json` must match the model's output index order.
- `q_hat` is specific to the `alpha` value used. If you change `alpha`, re-run the notebook.
- Calibration should be re-run whenever the model is updated.